## Numbers & Viz for Script 4

In [ ]:
import torch
from transformers import pipeline
from torch.nn import functional as F
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt

from transformers import GPT2LMHeadModel, GPT2Tokenizer, GPT2Config
from transformers import LlamaForCausalLM, PreTrainedTokenizerFast, LlamaConfig
from transformers import AutoModelForCausalLM, AutoTokenizer

device='cuda'

In [ ]:
model_id = "meta-llama/Llama-3.2-1B"
# model_id = "openai-community/gpt2"
# model_id = "google/gemma-3-1b-pt"
# model_id = "gpt2"

In [ ]:
# Random init
# model_config = GPT2Config() #Full sized model
# model = GPT2LMHeadModel(model_config).to(device) #Ok i should see what happens with full and pretrained model. 
# tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

# Not quite working for Llama, will take some noodling
# model_config  = LlamaConfig.from_dict(config_dict)
# model = LlamaForCausalLM(model_config).to(device) 
# tokenizer = AutoTokenizer.from_pretrained(model_id)

#Pretrained
model = AutoModelForCausalLM.from_pretrained(model_id).to(device)
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [ ]:
text = "The capital of France is Paris"
inputs = tokenizer(text, return_tensors="pt").to(device)
input_ids = inputs["input_ids"]

with torch.no_grad():
    outputs = model(input_ids, labels=input_ids)

my_probs=F.softmax(outputs.logits, dim=-1)
y_one_hot=F.one_hot(input_ids, num_classes=model.config.vocab_size)
correct_next_token_probs = (my_probs[:,:-1]*y_one_hot[:,1:]).sum(-1) #I'm sure there's waaay more efficient ways to do this
my_loss=-torch.log(correct_next_token_probs).mean()
print(my_loss.item(), outputs.loss.item())

Quick loss breakdown by token for paragraph 17b

In [ ]:
-torch.log(correct_next_token_probs)

In [ ]:
input_ids

In [ ]:
my_probs[0, 2, 315]

In [ ]:
prompt="The capital of France is"
input_ids=tokenizer.encode(prompt, return_tensors="pt")

In [ ]:
input_ids

In [ ]:
with torch.no_grad():
    out=model(input_ids.to(device))
probs = F.softmax(out.logits.detach().cpu(), dim=-1)

In [ ]:
sI=np.argsort(probs[0,-1, :].detach().cpu().float().numpy())[::-1]

for i in sI[:10]:
    print(i, round(probs[0, -1, i].item(),5), tokenizer.decode([i]))

Let’s start by visualizing the impact of just one of our one billion model parameters on our model’s output. The multilayer perceptron compute block in the model’s final layer has N total parameters, we’ll pick out one of these parameters, and see how it impacts our model’s final output. 

In [ ]:
model.model.layers[-1].mlp

In [ ]:
model.model.layers[-1].mlp.up_proj.weight.numel()

In [ ]:
8192*2048

In [ ]:
(2048*8192+2048*8192+8192*2048)/1e6

Our parameter’s current value is XX.YY. Let’s increase this parameter’s value by N, rerun our input text through our model, and see how our predictions and loss changes. Our model’s predicted probability of a final token of Paris moves down a little to 0.XX. Testing a change in the other direction, decreasing our parameters value to  0.0XA, moves up our model’s confidence to 0.XX. So for this single parameter and for this example text, we now know that we can make our model more confident in the right answer by decreasing this parameter’s value. 

In [ ]:
prompt2="The capital of France is Paris"
input_ids=tokenizer.encode(prompt2, return_tensors="pt")

out=model(input_ids.to(device), labels=input_ids.to(device))
out.loss.backward()

param_dict = {name: param for name, param in model.named_parameters()}
param_dict_2d={k:v for k,v in param_dict.items() if v.ndim>1}
grad_dict = {}

for p in param_dict:
    if param_dict[p].requires_grad:
        grad_dict[p] = param_dict[p].grad

grad_count=0
for g in grad_dict:
    grad_count+=grad_dict[g].numel()

grad_count/1e9

In [ ]:
input_ids

In [ ]:
def get_random_param_name(param_dict_2d):
    random_layer_index=np.random.choice(len(param_dict_2d.keys()))
    random_layer_name=list(param_dict_2d.keys())[random_layer_index]
    return random_layer_name

def get_random_indices(t):
    random_i=np.random.choice(t.shape[0])
    random_j=np.random.choice(t.shape[1])
    return random_i, random_j

def top_n_abs_indices(arr, n=128):
    flat_indices = np.argsort(np.abs(arr).flatten())[-n:][::-1]
    return np.column_stack(np.unravel_index(flat_indices, arr.shape))

In [ ]:
# [o[0] for o in model.named_parameters()]

- Ok so this is a litle messy, but I think the way to go, let's just measture the loss with respect to Paris

In [ ]:
with torch.no_grad():
    out=model(input_ids.to(device))
probs = F.softmax(out.logits.detach().cpu(), dim=-1)
my_loss=-np.log(probs[0, 5, 12366].item())

In [ ]:
my_loss.item()

In [ ]:
# p='model.layers.15.mlp.gate_proj.weight'
# p='model.layers.15.mlp.up_proj.weight'
p='model.layers.15.mlp.down_proj.weight'

In [ ]:
top_indices=top_n_abs_indices(grad_dict[p].detach().cpu().float().numpy())

In [ ]:
top_index=10 #55
n_stds=256
i_choice=top_indices[top_index][0]
j_choice=top_indices[top_index][1]

og_value=param_dict[p][i_choice, j_choice].item()
w_std=param_dict[p].detach().cpu().std().item()
print(og_value)

# xs=np.linspace(og_value-n_stds*w_std, og_value+n_stds*w_std, 31)
# xs=np.linspace(-1, 4, 255)
xs=np.arange(-1, 4, 0.01)
model.eval()
losses=[]; all_probs=[]
with torch.no_grad():
    for x in xs:
        param_dict[p][i_choice, j_choice]=x
        out=model(input_ids.to(device), labels=input_ids.to(device))
        # losses.append(out.loss.item())

        #Loss for just paris token
        probs = F.softmax(out.logits.detach().cpu(), dim=-1)
        my_loss=-np.log(probs[0, 5, 12366])
        losses.append(my_loss.item())
        all_probs.append(probs[0, 5, 12366].item())

    param_dict[p][i_choice, j_choice]=og_value

plt.plot(xs, losses)

In [ ]:
plt.plot(xs, losses)
plt.plot(xs, all_probs)
plt.savefig('p21_1.svg')

In [ ]:
i_choice, j_choice, p, og_value

In [ ]:
with torch.no_grad():
    param_dict[p][i_choice, j_choice]=og_value
    out=model(input_ids.to(device), labels=input_ids.to(device))
    probs = F.softmax(out.logits.detach().cpu(), dim=-1)
    my_loss=-np.log(probs[0, 5, 12366])
    param_dict[p][i_choice, j_choice]=og_value
    print(param_dict[p][i_choice, j_choice], probs[0, 5, 12366], my_loss)

In [ ]:
with torch.no_grad():
    param_dict[p][i_choice, j_choice]=og_value+0.01
    out=model(input_ids.to(device), labels=input_ids.to(device))
    probs = F.softmax(out.logits.detach().cpu(), dim=-1)
    my_loss=-np.log(probs[0, 5, 12366])
    print(param_dict[p][i_choice, j_choice])
    param_dict[p][i_choice, j_choice]=og_value
    print(probs[0, 5, 12366], my_loss)

In [ ]:
with torch.no_grad():
    param_dict[p][i_choice, j_choice]=og_value-0.01
    out=model(input_ids.to(device), labels=input_ids.to(device))
    probs = F.softmax(out.logits.detach().cpu(), dim=-1)
    print(param_dict[p][i_choice, j_choice])
    my_loss=-np.log(probs[0, 5, 12366])
    param_dict[p][i_choice, j_choice]=og_value
    print(probs[0, 5, 12366], my_loss)

In [ ]:
og_value

## Ok Now Whack a Mole

- Ok from here I think I need to go throug the whole exercise of picking and tweaking a second parameter, after "optimizing" the first parameter, I need to come back and show that the loss curve changed.

In [ ]:
top_index=10 #55
n_stds=256
i_choice=top_indices[top_index][0]
j_choice=top_indices[top_index][1]

og_value=param_dict[p][i_choice, j_choice].item()
w_std=param_dict[p].detach().cpu().std().item()

# xs1=np.linspace(og_value-n_stds*w_std, og_value+n_stds*w_std,255)
xs1=np.arange(-1.007, 4, 0.01)
model.eval()
losses1=[]; all_probs_1=[]
with torch.no_grad():
    for x in xs1:
        param_dict[p][i_choice, j_choice]=x
        out=model(input_ids.to(device), labels=input_ids.to(device))
        # losses.append(out.loss.item())

        #Loss for just paris token
        probs = F.softmax(out.logits.detach().cpu(), dim=-1)
        my_loss=-np.log(probs[0, 5, 12366])
        losses1.append(my_loss.item())
        all_probs_1.append(probs[0, 5, 12366])

    param_dict[p][i_choice, j_choice]=og_value

plt.plot(xs1, losses1)

In [ ]:
plt.plot(xs1, losses1, color='#ffd35a', linewidth=3)
plt.savefig('p24_1.svg')

In [ ]:
np.array(all_probs_1)

In [ ]:
i_choice, j_choice

In [ ]:
xs1[np.argmin(losses1)] #Ok wow yeah that's a nice big move. 

In [ ]:
losses[np.argmin(losses1)]

In [ ]:
all_probs[np.argmin(losses1)]

In [ ]:
with torch.no_grad():
    param_dict[p][i_choice, j_choice]=xs1[np.argmin(losses1)]  #Make a semi-presistent change here, hopefully this doesn't fuck up my other curve to much. 

In [ ]:
param_dict[p][i_choice, j_choice].item()

In [ ]:
top_index=12 #75 #55 4 is not terrible
n_stds=256
i_choice=top_indices[top_index][0]
j_choice=top_indices[top_index][1]

og_value=param_dict[p][i_choice, j_choice].item()
w_std=param_dict[p].detach().cpu().std().item()

# xs=np.linspace(og_value-n_stds*w_std, og_value+n_stds*w_std, 255)
xs=np.arange(-1, 4, 0.02)
model.eval()
losses=[]
with torch.no_grad():
    for x in tqdm(xs):
        param_dict[p][i_choice, j_choice]=x
        out=model(input_ids.to(device), labels=input_ids.to(device))
        # losses.append(out.loss.item())

        #Loss for just paris token
        probs = F.softmax(out.logits.detach().cpu(), dim=-1)
        my_loss=-np.log(probs[0, 5, 12366])
        losses.append(my_loss.item())

    param_dict[p][i_choice, j_choice]=og_value

plt.plot(xs, losses); plt.grid(1)

In [ ]:
plt.plot(xs, losses, color='#00ffff', linewidth=4);
plt.savefig('p24_2.svg')

In [ ]:
i_choice, j_choice

In [ ]:
xs[np.argmin(losses)]

In [ ]:
with torch.no_grad():
    param_dict[p][i_choice, j_choice]=xs[np.argmin(losses)]

In [ ]:
param_dict[p][i_choice, j_choice]

In [ ]:
#Back to first variable
top_index=10 #55
n_stds=256
i_choice=top_indices[top_index][0]
j_choice=top_indices[top_index][1]

og_value=param_dict[p][i_choice, j_choice].item()
w_std=param_dict[p].detach().cpu().std().item()

# xs2=np.linspace(og_value-n_stds*w_std, og_value+n_stds*w_std, 255)
xs2=np.arange(-1, 4, 0.02)
model.eval()
losses2=[]
with torch.no_grad():
    for x in xs2:
        param_dict[p][i_choice, j_choice]=x
        out=model(input_ids.to(device), labels=input_ids.to(device))
        # losses.append(out.loss.item())

        #Loss for just paris token
        probs = F.softmax(out.logits.detach().cpu(), dim=-1)
        my_loss=-np.log(probs[0, 5, 12366])
        losses2.append(my_loss.item())

    param_dict[p][i_choice, j_choice]=og_value

plt.plot(xs1, losses1)
plt.plot(xs2, losses2)

- Ok so great that this is working! I wish these were a little more different - let me noodle a little.
- Ok indices 0 and 4 are not terrible together, i would like a wider shift if I can have one. 

In [ ]:
plt.plot(xs1, losses1, c='#ffd35a', linewidth=4, alpha=0.5)
plt.plot(xs2, losses2, c='#ffd35a', linewidth=4)
plt.savefig('p24_3.svg')

In [ ]:
xs2[np.argmin(losses2)]

---

In [ ]:
out.logits.shape

In [ ]:
with torch.no_grad():
    out=model(input_ids.to(device))
probs = F.softmax(out.logits.detach().cpu(), dim=-1)
my_loss=-np.log(probs[0, 5, 12366].item())

In [ ]:
my_loss

In [ ]:
i_choice, j_choice, p, og_value

In [ ]:
for top_index in range(100):
    i_choice=top_indices[top_index][0]
    j_choice=top_indices[top_index][1]
    
    og_value=param_dict[p][i_choice, j_choice].item()
    print(top_index, i_choice, j_choice, og_value)